# 06 Three-Way Comparison: Detection & Explanation

**Pure Faster R-CNN** vs **Neuro-Symbolic (FRCNN + SODT)** vs **Grad-CAM**

| Aspect | Matchup | Question | Metrics |
|---|---|---|---|
| **Detection** | FRCNN vs NeSy | Does the tree cost accuracy? | mAP, Precision, Recall, F1 |
| **Explanation (4a)** | SODT (exact) vs GradCAM vs Random | Is SODT's explanation at least as faithful/grounded as GradCAM's? | Sufficiency, Necessity, Pointing, IoU |
| **Explanation (4b)** | SODT only, per node | Does the region each step weighs actually decide that step? | Necessity flip, Deletion/Insertion AUC — no GradCAM equivalent, it has no steps |

Per-image views: detection (FRCNN | NeSy | Truth), explanation (tree + per-step heatmap | GradCAM).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import random
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import time as _time
import torch

from notebooks.util import resolve_root
from neuro.config import NeuroConfig, NeuroTrainConfig
from neuro.inference import load_checkpoint_model
from neuro.prepare_dataset import PCBDataset
from neuro.preprocess_dataset import test_preprocess
from neurosym.inference import (
    load_neurosymbolic_detector,
    run_neurosymbolic_inference,
    explain_hybrid_detections,
    select_detection_indices,
)
from neurosym.visualization import (
    image_to_array,
    heatmap_to_array,
    draw_numbered_detections,
    draw_neurosymbolic_explanation,
    zoom_axis_to_box,
    draw_ground_truth_boxes,
    lookup_ground_truth,
)
from gradcam.gradcam import GradCAM
from util.config import load_yaml
from util.device import select_device

from torchvision.ops import box_iou as _box_iou
from tqdm import tqdm as _tqdm
from neuro.train import evaluate_model
from neuro.utils import detection_collate_fn
from torch.utils.data import DataLoader
from symbolic.evaluation import (
    evaluate_symbolic_model,
    evaluate_symbolic_spatial_metrics,
)
from gradcam.evaluation import evaluate_gradcam

PROJECT_ROOT = resolve_root()

## 1. Load Models

In [ ]:
neuro_config = load_yaml(Path("neuro.yaml"), NeuroConfig)
train_config = load_yaml(Path("neuro_train.yaml"), NeuroTrainConfig)

device = select_device(train_config["device"])
detector_ckpt = PROJECT_ROOT / "checkpoints" / "neuro" / "neuro_chosen.pt"
# The SODT checkpoint trained in notebooks/03 on the full trainval dump.
symbolic_ckpt = PROJECT_ROOT / "checkpoints" / "symbolic" / "symbolic_chosen_d6_l120_a0,15.pt"

class_names = tuple(train_config["dataset"]["class_names"])
preprocess = test_preprocess()

# --- Pure Faster R-CNN ---
frcnn_model, _ = load_checkpoint_model(
    detector_ckpt,
    model_config_path=Path("neuro.yaml"),
    train_config_path=Path("neuro_train.yaml"),
    device=str(device),
)
print("✓ Pure Faster R-CNN loaded")

# --- Neuro-Symbolic (SODT) ---
hybrid_model, _ = load_neurosymbolic_detector(
    detector_checkpoint_path=detector_ckpt,
    neuro_config=neuro_config,
    train_config=train_config,
    symbolic_checkpoint_path=symbolic_ckpt,
    device=str(device),
)
print("✓ Neuro-Symbolic (Faster R-CNN + SODT) loaded")

# --- Grad-CAM ---
gradcam = GradCAM(frcnn_model, device=str(device))
print("✓ Grad-CAM initialised (hooked into layer4/c5)")

## 2. Prepare Test Data

In [ ]:
from util.seed import seed_everything

test_dataset = PCBDataset(train_config, split_file="test.txt")
test_dataset.add_preprocess(preprocess)

N = 4  # Number of test images to compare
seed_everything(42)
sample_indices = random.sample(range(len(test_dataset)), k=min(N, len(test_dataset)))

print(f"Test set size : {len(test_dataset)}")
print(f"Selected indices: {sample_indices}")

## 3. Metrics Computation
Detection (FRCNN vs NeSy) first; explanation (4a head-to-head vs GradCAM, 4b per-step for SODT
only) after.

In [ ]:
from torchmetrics.detection import MeanAveragePrecision
from neuro.train import count_detection_matches, _confusion_and_per_class

SCORE_THRESHOLD = 0.3
MIN_PROPOSAL_IOU = 0.5  # GradCAM's own-resolution population below

# ── Detection Metrics: FRCNN (batched) ──
test_loader = DataLoader(
    test_dataset,
    batch_size=train_config["dataset"]["batch_size"],
    shuffle=False,
    collate_fn=detection_collate_fn,
)

print("Evaluating Faster R-CNN detection metrics...")
frcnn_det_metrics = evaluate_model(frcnn_model, test_loader, device, train_config)


# ── Single NeSy pass: detection metrics + explanation data ──

tree = hybrid_model.symbolic_tree
evaluation_config = train_config["evaluation"]
num_classes = len(class_names) + 1

# Detection accumulators
nesy_coco = MeanAveragePrecision(
    iou_type="bbox",
    backend="pycocotools",
    class_metrics=evaluation_config["class_metrics"],
)
nesy_paper = MeanAveragePrecision(
    iou_type="bbox",
    backend="pycocotools",
    iou_thresholds=evaluation_config["iou_thresholds"],
)
nesy_conf_matrix = torch.zeros(num_classes, num_classes, dtype=torch.int64)
nesy_class_tp = torch.zeros(num_classes, dtype=torch.int64)
nesy_class_fp = torch.zeros(num_classes, dtype=torch.int64)
nesy_class_fn = torch.zeros(num_classes, dtype=torch.int64)
nesy_tp = nesy_fp = nesy_fn = 0
nesy_forward_times = []

# Explanation accumulators
all_features = []
all_grids = []
all_proposal_boxes = []
all_matched_gt_boxes = []
all_has_matched = []
all_gt_iou = []
all_images = []
all_targets = []

print("Running single NeSy pass (detection + explanation)...")
for batch_images, batch_targets in _tqdm(test_loader, desc="NeSy combined eval"):
    t0 = _time.perf_counter()
    with torch.inference_mode():
        outputs = hybrid_model([img.to(device) for img in batch_images])
    nesy_forward_times.append(_time.perf_counter() - t0)

    for det, target in zip(outputs, batch_targets):
        # ── Detection metrics ──
        prediction = {
            "boxes": det["boxes"].detach().cpu(),
            "scores": det["scores"].detach().cpu(),
            "labels": det["labels"].detach().cpu(),
        }
        ground_truth = {
            "boxes": target["boxes"].detach().cpu(),
            "labels": target["labels"].detach().cpu(),
        }
        nesy_coco.update([prediction], [ground_truth])
        nesy_paper.update([prediction], [ground_truth])

        tp, fp, fn = count_detection_matches(
            prediction,
            ground_truth,
            iou_threshold=evaluation_config["precision_iou"],
            score_threshold=evaluation_config["precision_score_threshold"],
        )
        nesy_tp += tp
        nesy_fp += fp
        nesy_fn += fn
        pc = _confusion_and_per_class(
            prediction,
            ground_truth,
            iou_threshold=evaluation_config["precision_iou"],
            score_threshold=evaluation_config["precision_score_threshold"],
            num_classes=len(class_names),
        )
        nesy_conf_matrix += pc["confusion"]
        nesy_class_tp += pc["class_tp"]
        nesy_class_fp += pc["class_fp"]
        nesy_class_fn += pc["class_fn"]

        # ── Explanation data ──
        n_det = det["boxes"].shape[0]
        all_images.append(target.get("_image_tensor", None))  # placeholder
        all_targets.append(target)

        if n_det == 0:
            continue
        pooled = det["pooled_features"].detach().cpu()
        proposals = det["proposal_boxes"].detach().cpu()
        gt_boxes = target["boxes"]

        all_features.append(pooled.flatten(start_dim=1).numpy())
        all_grids.append(pooled.numpy())
        all_proposal_boxes.append(proposals)

        if gt_boxes.numel() > 0:
            overlaps = _box_iou(proposals, gt_boxes)
            matched_iou, matched_idx = overlaps.max(dim=1)
            matched_boxes = gt_boxes[matched_idx]
            has_match = matched_iou > 0
            all_matched_gt_boxes.append(matched_boxes)
            all_has_matched.append(has_match)
            all_gt_iou.append(matched_iou)
        else:
            all_matched_gt_boxes.append(torch.zeros((n_det, 4)))
            all_has_matched.append(torch.zeros(n_det, dtype=torch.bool))
            all_gt_iou.append(torch.zeros(n_det))

# ── Build NeSy detection metrics dict ──
nesy_coco_out = nesy_coco.compute()
nesy_paper_out = nesy_paper.compute()
nesy_precision = nesy_tp / max(nesy_tp + nesy_fp, 1)
nesy_recall = nesy_tp / max(nesy_tp + nesy_fn, 1)
all_class_names = ["__background__", *class_names]

nesy_det_metrics = {
    "mAP@0.5:0.95": nesy_coco_out["map"].item(),
    "mAP@0.5": nesy_coco_out["map_50"].item(),
    "AP75": nesy_coco_out["map_75"].item(),
    "AP@50:5:85": nesy_paper_out["map"].item(),
    "precision": nesy_precision,
    "recall": nesy_recall,
    "mar_100": nesy_coco_out["mar_100"].item(),
    "f1_score": 2
    * nesy_precision
    * nesy_recall
    / max(nesy_precision + nesy_recall, 1e-10),
    "inference_time_ms": float(np.mean(nesy_forward_times)) * 1000
    if nesy_forward_times
    else 0.0,
    "confusion_matrix": nesy_conf_matrix.tolist(),
    "confusion_matrix_labels": all_class_names,
}

print(
    f"FRCNN  mAP@0.5:0.95 = {frcnn_det_metrics['mAP@0.5:0.95']:.4f}  |  NeSy  mAP@0.5:0.95 = {nesy_det_metrics['mAP@0.5:0.95']:.4f}"
)
print(
    f"FRCNN  precision    = {frcnn_det_metrics['precision']:.4f}  |  NeSy  precision    = {nesy_det_metrics['precision']:.4f}"
)
print(
    f"FRCNN  recall       = {frcnn_det_metrics['recall']:.4f}  |  NeSy  recall       = {nesy_det_metrics['recall']:.4f}"
)

# ── Explanation data assembled (used by 4a/4b below) ──
feature_matrix = np.concatenate(all_features, axis=0).astype(np.float32)
feature_grids = torch.from_numpy(np.concatenate(all_grids, axis=0))
all_proposals_t = torch.cat(all_proposal_boxes, dim=0)
all_matched_gt_t = torch.cat(all_matched_gt_boxes, dim=0)
all_has_matched_t = torch.cat(all_has_matched, dim=0)
all_gt_iou_t = torch.cat(all_gt_iou, dim=0)

nesy_pred = tree.predict(feature_matrix)

# Sufficiency stays on the pooled 7x7 grid — it was never migrated to the
# FPN-native protocol (see 4a: it's not discriminative either way), plus a
# random-ranking control on the identical masking budget.
nesy_sufficiency = evaluate_symbolic_model(
    tree, feature_matrix, nesy_pred, class_names, compute_auc=False
)  # AUC unused below (FPN-native per-node AUC covers it instead) — skip the expensive pass
nesy_sufficiency_random = evaluate_symbolic_model(
    tree, feature_matrix, nesy_pred, class_names, ranking="random", compute_auc=False
)

# ── GradCAM explanation metrics (needs images; collect separately) ──
print("Computing GradCAM explanation metrics...")
gradcam_images = []
gradcam_targets = []
for i in range(len(test_dataset)):
    image, target = test_dataset[i]
    gradcam_images.append(image)
    gradcam_targets.append(target)

gradcam_expl_metrics = evaluate_gradcam(
    model=frcnn_model,
    gradcam=gradcam,
    images=gradcam_images,
    targets=gradcam_targets,
    score_threshold=SCORE_THRESHOLD,
    num_images=500,
    min_proposal_iou=MIN_PROPOSAL_IOU,
)

# ── Inference Time Summary ──
print(f"\n{'─' * 60}")
print("Average Inference Time per Image (metrics evaluation)")
print(f"{'─' * 60}")
print(
    f"  Faster R-CNN : {frcnn_det_metrics['inference_time_ms']:.1f} ms  (batched avg)"
)
print(
    f"  NeSy (FRCNN+SODT) : {nesy_det_metrics['inference_time_ms']:.1f} ms  (batched avg)"
)
print(
    f"  GradCAM (FRCNN+GradCAM) : {gradcam_expl_metrics['inference_time_ms_avg']:.1f} ms  (avg)"
)
print(
    f"    GradCAM min: {gradcam_expl_metrics['inference_time_ms_min']:.1f} ms  max: {gradcam_expl_metrics['inference_time_ms_max']:.1f} ms"
)


## 4. Metrics Visualization

Detection first (FRCNN vs NeSy). Explanation is two tables: a head-to-head against GradCAM
(4a — each method scored at its own resolution, `leaf_only` dropped from reporting), then a
per-step breakdown that only SODT's six-question structure can answer (4b).

In [ ]:
# ── Detection Metrics Bar Chart + Confusion Matrix ──
det_keys = ["mAP@0.5:0.95", "mAP@0.5", "precision", "recall", "f1_score"]
det_labels = ["mAP@0.5:0.95", "mAP@0.5", "Precision", "Recall", "F1 Score"]
frcnn_vals = [frcnn_det_metrics[k] for k in det_keys]
nesy_vals = [nesy_det_metrics[k] for k in det_keys]

fig = plt.figure(figsize=(18, 6))
gs = fig.add_gridspec(1, 3, width_ratios=[2.2, 1, 1], wspace=0.35)

# Panel 1: Bar chart
ax = fig.add_subplot(gs[0, 0])
x = np.arange(len(det_labels))
w = 0.35
b1 = ax.bar(
    x - w / 2, frcnn_vals, w, label="Faster R-CNN", color="#2F6F9F", edgecolor="white"
)
b2 = ax.bar(
    x + w / 2,
    nesy_vals,
    w,
    label="NeSy (FRCNN+SODT)",
    color="#D97941",
    edgecolor="white",
)
ax.bar_label(b1, fmt="%.3f", padding=3, fontsize=8)
ax.bar_label(b2, fmt="%.3f", padding=3, fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(det_labels, fontsize=10)
ax.set_ylabel("Score")
ax.set_title("Detection Metrics: FRCNN vs NeSy", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_ylim(0, 1.15)
ax.grid(axis="y", alpha=0.2)


# Panel 2: FRCNN confusion matrix
def _plot_confusion_matrix(ax, metrics_dict, title):
    cm = np.array(metrics_dict["confusion_matrix"])
    labels = metrics_dict["confusion_matrix_labels"]
    # Include background row/col (index 0)
    cm_defect = cm
    defect_labels = labels
    # Normalize per row (recall-normalized)
    row_sums = cm_defect.sum(axis=1, keepdims=True)
    cm_norm = np.where(row_sums > 0, cm_defect / row_sums, 0.0)

    ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(defect_labels)))
    ax.set_yticks(range(len(defect_labels)))
    ax.set_xticklabels(defect_labels, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(defect_labels, fontsize=7)
    for i in range(cm_defect.shape[0]):
        for j in range(cm_defect.shape[1]):
            color = "white" if cm_norm[i, j] > 0.5 else "black"
            ax.text(
                j,
                i,
                f"{cm_defect[i, j]}\n({cm_norm[i, j]:.2f})",
                ha="center",
                va="center",
                fontsize=6,
                color=color,
            )
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")


ax_frcnn_cm = fig.add_subplot(gs[0, 1])
_plot_confusion_matrix(ax_frcnn_cm, frcnn_det_metrics, "FRCNN Confusion Matrix")

# Panel 3: NeSy confusion matrix
ax_nesy_cm = fig.add_subplot(gs[0, 2])
_plot_confusion_matrix(ax_nesy_cm, nesy_det_metrics, "NeSy Confusion Matrix")

plt.tight_layout()
plt.show()

## 4a. Explanation Faithfulness & Localization: SODT (exact) vs GradCAM vs Random
Each method masks its own resolution's top 50% (`_FAITHFULNESS_CELL_BUDGET_FRACTION`) — SODT's
exact FPN map, GradCAM's own map — never shrunk to a shared coarse grid first
(`neurosym/evaluation.py::evaluate_faithfulness_fpn_masking`). `leaf_only` (the old 7x7-grid
SODT ranking) is dropped from every row here: it handicapped SODT to a coarser grid than the
exact map actually has.

- **Necessity flip** and **Pointing/IoU** (loose proposals, `0.05–0.35` — tight proposals give
  n=0, GT fills the grid) are head-to-head, random-baselined.
- **Sufficiency** is reported, not relied on — a random ranking scores about as well (see the
  random column): zeroing ~40/49 cells pushes routing off-distribution regardless of which
  cells are picked, so this metric doesn't discriminate on an oblique tree.
- **Deletion/Insertion AUC** is NOT head-to-head here. GradCAM's own AUC is a class-probability
  curve; SODT's only exists per-node (4b) as a routing-confidence curve — different quantities,
  printed separately rather than forced into one bar.
- **Population caveat**: SODT's necessity/sufficiency run on their own live-pass populations
  (raw RPN proposals / all detections); GradCAM's use `min_proposal_iou`-filtered ones. Same
  fraction-of-own-resolution masking budget, not the same exact RoI set — directional, not
  decimal-precise, until unified.

In [ ]:
from neurosym.evaluation import (
    evaluate_faithfulness_fpn_masking,
    evaluate_exact_attribution_spatial_metrics,
)
from util.heatmap_metrics import evaluate_random_baseline_spatial_metrics

LOOSE_MIN_IOU = 0.05
LOOSE_MAX_IOU = 0.35
# Per-node (needed for 4b below) is ~2 orders of magnitude more re-pools than
# path-level alone — this one FPN-masking pass covers BOTH tables, so it's
# not recomputed in 4b. Time images[:5] before raising this subsample.
FAITHFULNESS_FPN_SUBSAMPLE = 60
EXACT_ATTR_SUBSAMPLE = 150  # live forward pass per RoI; capped for runtime.

fpn_faith_images = [test_dataset[i][0] for i in range(FAITHFULNESS_FPN_SUBSAMPLE)]
fpn_faithfulness = evaluate_faithfulness_fpn_masking(hybrid_model, fpn_faith_images)

gcam_loose = evaluate_gradcam(
    model=frcnn_model,
    gradcam=gradcam,
    images=gradcam_images,
    targets=gradcam_targets,
    score_threshold=SCORE_THRESHOLD,
    num_images=500,
    min_proposal_iou=LOOSE_MIN_IOU,
    max_proposal_iou=LOOSE_MAX_IOU,
)
random_loose = evaluate_random_baseline_spatial_metrics(
    all_proposals_t,
    all_matched_gt_t,
    all_has_matched_t,
    all_gt_iou_t,
    min_proposal_iou=LOOSE_MIN_IOU,
    max_proposal_iou=LOOSE_MAX_IOU,
)

print(f"Computing exact-attribution spatial metrics on {EXACT_ATTR_SUBSAMPLE} images...")
exact_attr_images = [test_dataset[i][0] for i in range(EXACT_ATTR_SUBSAMPLE)]
exact_attr_targets = [test_dataset[i][1] for i in range(EXACT_ATTR_SUBSAMPLE)]
exact_attr_loose = evaluate_exact_attribution_spatial_metrics(
    hybrid_model,
    exact_attr_images,
    exact_attr_targets,
    min_proposal_iou=LOOSE_MIN_IOU,
    max_proposal_iou=LOOSE_MAX_IOU,
)

sodt_necessity = fpn_faithfulness["path"]["exact"]["necessity_prediction_flip_rate"]
random_necessity = fpn_faithfulness["path"]["random"]["necessity_prediction_flip_rate"]

print(f"\n{'Metric':<28} {'SODT':>10} {'GradCAM':>10} {'Random':>10}")
print("-" * 60)
print(
    f"{'Necessity flip':<28} {sodt_necessity:>10.4f} "
    f"{gradcam_expl_metrics['necessity_prediction_flip_rate']:>10.4f} {random_necessity:>10.4f}"
)
print(
    f"{'Sufficiency (own ranking)':<28} "
    f"{nesy_sufficiency['sufficiency_prediction_preservation']:>10.4f} "
    f"{gradcam_expl_metrics['sufficiency_prediction_preservation']:>10.4f} "
    f"{nesy_sufficiency_random['sufficiency_prediction_preservation']:>10.4f}"
)
print(
    f"{'Pointing (loose)':<28} {exact_attr_loose['pointing_score']:>10.4f} "
    f"{gcam_loose['pointing_score']:>10.4f} {random_loose['pointing_score']:>10.4f}"
)
print(
    f"{'IoU overlap (loose)':<28} {exact_attr_loose['box_grounded_roi_overlap']:>10.4f} "
    f"{gcam_loose['box_grounded_roi_overlap']:>10.4f} {random_loose['box_grounded_roi_overlap']:>10.4f}"
)
print(
    f"{'n (loose)':<28} {exact_attr_loose['evaluated_roi_count']:>10,d} "
    f"{gcam_loose['evaluated_roi_count']:>10,d} {random_loose['evaluated_roi_count']:>10,d}"
)

# Same saturation problem WHAT-I-DID Sec 6.4 found — this subset (GT < 50% of
# grid) is where pointing/IoU still discriminate, on top of the loose-proposal
# population above.
exact_low = exact_attr_loose["low_gt_coverage"]
gcam_low = gcam_loose["low_gt_coverage"]
random_low = random_loose["low_gt_coverage"]
print(f"\n{'Metric (low GT coverage)':<28} {'SODT':>10} {'GradCAM':>10} {'Random':>10}")
print("-" * 60)
print(
    f"{'Pointing':<28} {exact_low['pointing_score']:>10.4f} "
    f"{gcam_low['pointing_score']:>10.4f} {random_low['pointing_score']:>10.4f}"
)
print(
    f"{'IoU overlap':<28} {exact_low['box_grounded_roi_overlap']:>10.4f} "
    f"{gcam_low['box_grounded_roi_overlap']:>10.4f} {random_low['box_grounded_roi_overlap']:>10.4f}"
)
print(
    f"{'n':<28} {exact_low['evaluated_roi_count']:>10,d} "
    f"{gcam_low['evaluated_roi_count']:>10,d} {random_low['evaluated_roi_count']:>10,d}"
)
print(
    f"\nGradCAM's own deletion/insertion AUC (own resolution, probability-based, "
    f"no path-level SODT equivalent — SODT's AUC is per-node only, see 4b): "
    f"del={gradcam_expl_metrics['deletion_auc']:.4f} ins={gradcam_expl_metrics['insertion_auc']:.4f}"
)

# ── Chart ──
faith_labels = ["Sufficiency\n(own ranking)", "Necessity\nFlip Rate"]
sodt_faith = [nesy_sufficiency["sufficiency_prediction_preservation"], sodt_necessity]
gcam_faith = [
    gradcam_expl_metrics["sufficiency_prediction_preservation"],
    gradcam_expl_metrics["necessity_prediction_flip_rate"],
]

spatial_labels = ["Pointing\n(loose)", "IoU Overlap\n(loose)"]
sodt_spatial = [exact_attr_loose["pointing_score"], exact_attr_loose["box_grounded_roi_overlap"]]
gcam_spatial = [gcam_loose["pointing_score"], gcam_loose["box_grounded_roi_overlap"]]
random_spatial = [random_loose["pointing_score"], random_loose["box_grounded_roi_overlap"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    "Explanation Faithfulness & Localization: SODT (exact) vs GradCAM vs Random",
    fontsize=13, fontweight="bold", y=1.02,
)

ax = axes[0]
x = np.arange(len(faith_labels))
w = 0.35
b1 = ax.bar(x - w / 2, sodt_faith, w, label="SODT (exact)", color="#2F6F9F", edgecolor="white")
b2 = ax.bar(x + w / 2, gcam_faith, w, label="GradCAM", color="#D97941", edgecolor="white")
ax.bar_label(b1, fmt="%.3f", padding=3, fontsize=8, fontweight="bold")
ax.bar_label(b2, fmt="%.3f", padding=3, fontsize=8, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(faith_labels, fontsize=9)
ax.set_ylabel("Score")
ax.set_title("Faithfulness", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.set_ylim(0, 1.15)
ax.grid(axis="y", alpha=0.2)

ax = axes[1]
x = np.arange(len(spatial_labels))
w = 0.27
b1 = ax.bar(x - w, sodt_spatial, w, label="SODT (exact)", color="#2F6F9F", edgecolor="white")
b2 = ax.bar(x, gcam_spatial, w, label="GradCAM", color="#D97941", edgecolor="white")
b3 = ax.bar(x + w, random_spatial, w, label="Random baseline", color="#B0B0B0", edgecolor="white")
ax.bar_label(b1, fmt="%.3f", padding=3, fontsize=7, fontweight="bold")
ax.bar_label(b2, fmt="%.3f", padding=3, fontsize=7, fontweight="bold")
ax.bar_label(b3, fmt="%.3f", padding=3, fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels(spatial_labels, fontsize=9)
ax.set_ylabel("Score")
ax.set_title("Localization (loose proposals)", fontsize=12, fontweight="bold")
ax.legend(fontsize=8)
ax.set_ylim(0, 1.15)
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()


## 4b. Per-Step Faithfulness: Does Each Node's Own Region Decide That Node's Question?
Reuses `fpn_faithfulness` from 4a — no second FPN-masking pass. For each node on the path:
mask that node's own map, re-pool, check whether THAT NODE's own routing sign flips, plus a
deletion/insertion AUC over that node's routing confidence (`sigmoid(|w.x+b|)`, the same
quantity `NeuroSymbolicDetector` uses for routing-margin scoring — not a class probability,
so it's not comparable to GradCAM's AUC above).

No Grad-CAM column: Grad-CAM has no per-step structure to test against — this table is the
direct evidence for the six-panel heatmap figures in §5 ("this step weighs this region, and
that region is what decided this step"), not a lokalisasi claim.

In [ ]:
import math

print("Per-node — does the region this node weighs decide this node's own question?")
print(f"{'ranking':10} {'depth':>6} {'nec_flip':>10} {'ceiling':>9} {'support':>9} {'del_auc':>9} {'ins_auc':>9} {'n':>6}")
print("-" * 76)
ceiling = fpn_faithfulness["node_necessity_ceiling"]
for name, per_depth in fpn_faithfulness["node"].items():
    for depth, r in per_depth.items():
        # Ceiling is "exact"-specific (full-mask bound) — only meaningful
        # against the "exact" row, and only tight when support <= 0.5 (its
        # ranking then covers the node's whole nonzero support). "random"
        # has no "support" (no map of its own), shown as "—".
        support = f"{'—':>9}" if math.isnan(r['support_fraction']) else f"{r['support_fraction']:9.4f}"
        print(
            f"{name:10} {depth + 1:>6} {r['necessity_prediction_flip_rate']:10.4f} "
            f"{ceiling.get(depth, float('nan')):9.4f} {support} "
            f"{r['deletion_auc']:9.4f} {r['insertion_auc']:9.4f} {r['evaluated_roi_count']:6d}"
        )
print(
    "(ceiling: max flip rate 'exact' can reach if it masks the whole box — not a bound "
    "for 'random'. support: fraction of the box with any contribution — ceiling is only "
    "tight for 'exact' when this is <= 0.5, the masking budget.)"
)

sim = fpn_faithfulness["node_map_similarity"]
print(
    f"\nMean cosine similarity between consecutive nodes' maps: "
    f"{sim['mean_cosine_similarity_between_consecutive_nodes']:.4f} "
    f"(n={sim['evaluated_pair_count']:,}) — low means each step weighs a different region."
)


## 5. Per-Image Comparison
Per test image: detection (FRCNN | NeSy | Truth), then explanation. The tree panel shows the
path SODT took to its decision; each heatmap next to it shows the region THAT step weighed —
read top to bottom as a six-step (or however deep the path is) trace, not one summary map.
GradCAM sits alongside for the same detection, at its own resolution.

In [ ]:
MAX_DETECTIONS = 6

frcnn_times = []
nesy_times = []
gradcam_times = []


def _draw_gradcam_panel_in_axis(ax, image_tensor, gcam_result, class_names):
    from neurosym.heatmap import project_heatmap_to_image

    img = image_to_array(image_tensor)
    ax.imshow(img)

    dimmer = np.zeros((img.shape[0], img.shape[1], 4), dtype=np.float32)
    dimmer[..., 3] = 0.4
    ax.imshow(dimmer)

    heatmap_tensor = gcam_result["heatmap"]
    box = gcam_result["box"]
    image_shape = tuple(image_tensor.shape[-2:])

    projected = project_heatmap_to_image(heatmap_tensor, box, image_shape)

    rgba = heatmap_to_array(projected)
    ax.imshow(rgba)

    x1, y1, x2, y2 = box.tolist()
    ax.add_patch(
        patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor="cyan",
            facecolor="none",
        )
    )

    zoom_axis_to_box(ax, box, image_shape)
    label = gcam_result["label"]
    score = gcam_result["score"]
    name = class_names[label - 1] if 0 < label <= len(class_names) else f"cls{label}"
    ax.set_title(
        f"Grad-CAM Focus\n{name} {score:.2f}",
        fontsize=10,
        fontweight="bold",
        color="#c62828",
    )
    ax.axis("off")


def _draw_gt_panel(ax, image_tensor, target, class_names):
    """Draw ground-truth annotations on the image."""
    img = image_to_array(image_tensor)
    ax.imshow(img)
    gt_boxes = target["boxes"]
    gt_labels = target["labels"]
    palette = ["#d62828", "#0077b6", "#2a9d8f", "#f4a261", "#6a4c93", "#5f6f52"]
    for i in range(gt_boxes.shape[0]):
        x1, y1, x2, y2 = gt_boxes[i].tolist()
        lbl = int(gt_labels[i])
        name = class_names[lbl - 1] if 0 < lbl <= len(class_names) else f"cls{lbl}"
        color = palette[(lbl - 1) % len(palette)]
        ax.add_patch(
            patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2.5,
                edgecolor=color,
                facecolor="none",
            )
        )
        ax.text(
            x1,
            max(y1 - 4, 10),
            name,
            fontsize=8,
            fontweight="bold",
            color="white",
            backgroundcolor=color,
            alpha=0.85,
        )
    ax.set_title(
        f"Ground Truth ({gt_boxes.shape[0]} defects)",
        fontsize=12,
        fontweight="bold",
        color="#228B22",
    )
    ax.axis("off")


for sample_idx in sample_indices:
    image_tensor, target = test_dataset[sample_idx]
    image_shape = tuple(image_tensor.shape[-2:])

    # ---------- (A) Pure Faster R-CNN inference ----------
    t0 = _time.perf_counter()
    with torch.inference_mode():
        frcnn_preds = frcnn_model([image_tensor.to(device)])
        frcnn_det = {k: v.detach().cpu() for k, v in frcnn_preds[0].items()}
    t_frcnn = (_time.perf_counter() - t0) * 1000
    frcnn_times.append(t_frcnn)

    # ---------- (B) Neuro-Symbolic inference + explanation ----------
    t0 = _time.perf_counter()
    with torch.inference_mode():
        hybrid_preds = run_neurosymbolic_inference(
            hybrid_model, [image_tensor.to(device)]
        )
        hybrid_det = hybrid_preds[0]
    t_nesy = (_time.perf_counter() - t0) * 1000
    nesy_times.append(t_nesy)

    sodt_indices = select_detection_indices(
        hybrid_det,
        score_threshold=SCORE_THRESHOLD,
        max_detections=MAX_DETECTIONS,
    )

    sodt_explanations = explain_hybrid_detections(
        hybrid_model,
        hybrid_det,
        image_shape=image_shape,
        detection_indices=sodt_indices,
        mode="local_instance_evidence_map",
    )

    # ---------- (C) Grad-CAM ----------
    t0 = _time.perf_counter()
    boxes_selected = hybrid_det["boxes"][sodt_indices]
    labels_selected = hybrid_det["labels"][sodt_indices]
    scores_selected = hybrid_det["scores"][sodt_indices]

    if len(sodt_indices) > 0:
        gcam_heatmaps = gradcam.generate(
            image_tensor,
            boxes_selected.to(device),
            labels_selected.to(device),
            output_size=(7, 7),
        )
        gcam_results = [
            {
                "box": boxes_selected[i].detach().cpu(),
                "label": int(labels_selected[i]),
                "score": float(scores_selected[i]),
                "heatmap": gcam_heatmaps[i],
            }
            for i in range(len(sodt_indices))
        ]
    else:
        gcam_results = []
    t_gradcam = (_time.perf_counter() - t0) * 1000
    gradcam_times.append(t_gradcam)

    total_ms = t_frcnn + t_nesy + t_gradcam
    print(f"\n{'=' * 100}")
    print(
        f"Test Image #{sample_idx}  (FRCNN: {t_frcnn:.1f} | NeSy: {t_nesy:.1f} | GradCAM: {t_gradcam:.1f} | Total: {total_ms:.1f} ms)"
    )
    print(f"{'=' * 100}\n")

    # ═══════════════════════════════════════════════════════════
    # DETECTION SECTION: FRCNN | NeSy | Ground Truth
    # ═══════════════════════════════════════════════════════════
    print("── Detection ──")
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    draw_numbered_detections(
        axes[0], image_tensor, frcnn_det, sodt_indices, class_names
    )
    axes[0].set_title("Faster R-CNN Detections", fontsize=13, fontweight="bold")

    draw_numbered_detections(
        axes[1], image_tensor, hybrid_det, sodt_indices, class_names
    )
    axes[1].set_title("NeSy (FRCNN+SODT) Detections", fontsize=13, fontweight="bold")

    _draw_gt_panel(axes[2], image_tensor, target, class_names)

    plt.tight_layout()
    plt.show()

    # ═══════════════════════════════════════════════════════════
    # EXPLANATION SECTION: SODT Tree+Heatmap | GradCAM Heatmap
    # ═══════════════════════════════════════════════════════════
    print("── Explanation ──")
    if len(sodt_indices) == 0:
        print("No detections above threshold.")
        continue

    out = widgets.Output()

    def make_btn(
        btn_num,
        det_idx,
        expl,
        gcam_res,
        tnsr=image_tensor,
        hyb=hybrid_det,
        out_w=out,
    ):
        label_name = class_names[expl["label"] - 1]
        score = expl["score"]
        btn = widgets.Button(
            description=f"#{btn_num} {label_name} {score:.2f}",
            layout=widgets.Layout(width="180px"),
        )

        def on_click(_):
            with out_w:
                clear_output(wait=True)

                def gcam_panel(ax):
                    _draw_gradcam_panel_in_axis(ax, tnsr, gcam_res, class_names)

                draw_neurosymbolic_explanation(
                    tnsr,
                    hyb,
                    det_idx,
                    expl,
                    class_names,
                    hybrid_model.symbolic_tree,
                    selected_number=btn_num,
                    extra_panel_func=gcam_panel,
                )

        btn.on_click(on_click)
        return btn

    buttons = []
    for btn_num, (det_idx, expl, gcam_res) in enumerate(
        zip(sodt_indices, sodt_explanations, gcam_results), 1
    ):
        buttons.append(make_btn(btn_num, det_idx, expl, gcam_res))

    display(widgets.HBox(buttons))
    display(out)

# ── Inference Time Summary ──
if frcnn_times:

    def _print_stats(name, times):
        arr = np.array(times)
        print(
            f"  {name:<25s}  avg: {arr.mean():.1f} ms  min: {arr.min():.1f} ms  max: {arr.max():.1f} ms"
        )

    print(f"\n{'─' * 80}")
    print("Per-Image Inference Time (ms)")
    print(f"{'─' * 80}")
    _print_stats("Faster R-CNN", frcnn_times)
    _print_stats("NeSy (FRCNN+SODT)", nesy_times)
    _print_stats("GradCAM (FRCNN+GradCAM)", gradcam_times)
    totals = [f + n + g for f, n, g in zip(frcnn_times, nesy_times, gradcam_times)]
    _print_stats("TOTAL", totals)
    print(f"{'─' * 80}")
    print(f"{'Image':<10s} {'FRCNN':>8s} {'NeSy':>8s} {'GradCAM':>8s} {'Total':>8s}")
    for idx, tf, tn, tg in zip(sample_indices, frcnn_times, nesy_times, gradcam_times):
        print(f"  #{idx:<8d} {tf:>8.1f} {tn:>8.1f} {tg:>8.1f} {tf + tn + tg:>8.1f}")
else:
    print("No images processed.")

## 6. Interactive Comparison (Upload Image)
Upload runs all three pipes (FRCNN · NeSy · GradCAM): detection (FRCNN | NeSy),
explanation per detection (tree + heatmap | GradCAM).

In [ ]:
# ── Interactive Three-Way Comparison on Uploaded Image ──
# Upload a PCB image → automatically runs Pure FRCNN · NeSy (FRCNN+SODT) · GradCAM.
# Shows detection comparison and per-detection explanations.
# Cleanup: close all prior figures and clear outputs on every cell re-run.

import io
from PIL import Image
from neuro.inference import run_inference
from neurosym.heatmap import project_heatmap_to_image

plt.close("all")
clear_output(wait=True)

INT_SCORE_THRESHOLD = 0.3
INT_MAX_DETECTIONS = 15

int_upload = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload PCB Image",
)
int_output = widgets.Output()
expl_output = widgets.Output()
_processing_upload = False


def _int_file_record():
    value = int_upload.value
    if isinstance(value, tuple):
        return value[0] if value else None
    if isinstance(value, dict):
        if "content" in value:
            return value
        return next(iter(value.values())) if value else None
    return None


def _int_content_bytes(file_record) -> bytes:
    content = file_record["content"]
    return content.tobytes() if isinstance(content, memoryview) else bytes(content)


def _int_draw_gradcam_panel(ax, image_tensor, gcam_result, class_names):
    """Draw a single GradCAM heatmap overlay, zoomed to the detection box."""
    img = image_to_array(image_tensor)
    ax.imshow(img)
    dimmer = np.zeros((img.shape[0], img.shape[1], 4), dtype=np.float32)
    dimmer[..., 3] = 0.4
    ax.imshow(dimmer)
    heatmap_tensor = gcam_result["heatmap"]
    box = gcam_result["box"]
    image_shape = tuple(image_tensor.shape[-2:])
    projected = project_heatmap_to_image(heatmap_tensor, box, image_shape)
    rgba = heatmap_to_array(projected)
    ax.imshow(rgba)
    x1, y1, x2, y2 = box.tolist()
    ax.add_patch(
        patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            edgecolor="cyan",
            facecolor="none",
        )
    )
    zoom_axis_to_box(ax, box, image_shape)
    label = gcam_result["label"]
    score = gcam_result["score"]
    name = class_names[label - 1] if 0 < label <= len(class_names) else f"cls{label}"
    ax.set_title(
        f"GradCAM: {name} {score:.2f}",
        fontsize=11,
        fontweight="bold",
        color="cyan",
    )
    ax.axis("off")


def _on_upload_change(change):
    global _processing_upload
    if _processing_upload:
        return
    if not change.get("new"):
        return
    _processing_upload = True
    try:
        with int_output:
            from IPython.display import clear_output

            clear_output(wait=True)
            expl_output.clear_output(wait=True)
            plt.close("all")

            file_record = _int_file_record()
            if file_record is None:
                print("Upload a PCB image first.")
                return

            image_name = file_record.get("name", "uploaded_image")
            pil_image = Image.open(io.BytesIO(_int_content_bytes(file_record))).convert(
                "RGB"
            )
            image_tensor = preprocess(pil_image)
            image_shape = tuple(image_tensor.shape[-2:])
            print(f"Image: {image_name}")

            # ═══════════════════════════════════════════
            # (A) Pure Faster R-CNN
            # ═══════════════════════════════════════════
            t0 = _time.perf_counter()
            frcnn_det = run_inference(frcnn_model, [image_tensor], device)[0]
            t_frcnn = (_time.perf_counter() - t0) * 1000

            # ═══════════════════════════════════════════
            # (B) NeSy (FRCNN + SODT)
            # ═══════════════════════════════════════════
            t0 = _time.perf_counter()
            hybrid_det = run_neurosymbolic_inference(hybrid_model, [image_tensor])[0]
            t_nesy = (_time.perf_counter() - t0) * 1000

            sel_indices = select_detection_indices(
                hybrid_det,
                score_threshold=INT_SCORE_THRESHOLD,
                max_detections=INT_MAX_DETECTIONS,
            )
            sel_explanations = explain_hybrid_detections(
                hybrid_model,
                hybrid_det,
                image_shape=image_shape,
                detection_indices=sel_indices,
                mode="local_instance_evidence_map",
            )

            # ═══════════════════════════════════════════
            # (C) Grad-CAM (using NeSy detection boxes)
            # ═══════════════════════════════════════════
            t0 = _time.perf_counter()
            boxes_sel = hybrid_det["boxes"][sel_indices]
            labels_sel = hybrid_det["labels"][sel_indices]
            if len(sel_indices) > 0:
                from gradcam.gradcam import GradCAM

                local_gradcam = GradCAM(frcnn_model, device=str(device))
                try:
                    gcam_heatmaps = local_gradcam.generate(
                        image_tensor,
                        boxes_sel.to(device),
                        labels_sel.to(device),
                        output_size=(7, 7),
                    )
                finally:
                    local_gradcam.release()
                gcam_results = [
                    {
                        "box": boxes_sel[j].detach().cpu(),
                        "label": int(labels_sel[j]),
                        "score": float(hybrid_det["scores"][sel_indices[j]]),
                        "heatmap": gcam_heatmaps[j],
                    }
                    for j in range(len(sel_indices))
                ]
            else:
                gcam_results = []
            t_gradcam = (_time.perf_counter() - t0) * 1000

            total_ms = t_frcnn + t_nesy + t_gradcam
            print(
                f"FRCNN: {t_frcnn:.1f} ms | NeSy: {t_nesy:.1f} ms "
                f"| GradCAM: {t_gradcam:.1f} ms | Total: {total_ms:.1f} ms"
            )
            print(f"Detections above threshold: {len(sel_indices)}\n")

            # ═══════════════════════════════════════════
            # (GT) Ground Truth Lookup
            # ═══════════════════════════════════════════
            gt_info = lookup_ground_truth(test_dataset, image_name)

            # ═══════════════════════════════════════════
            # DETECTION SECTION: FRCNN | NeSy | Ground Truth
            # ═══════════════════════════════════════════
            print("── Detection ──")
            if gt_info is not None:
                gt_boxes, gt_labels = gt_info
                fig, axes = plt.subplots(1, 3, figsize=(20, 6))

                draw_numbered_detections(
                    axes[0], image_tensor, frcnn_det, sel_indices, class_names
                )
                axes[0].set_title(
                    "Faster R-CNN Detections", fontsize=13, fontweight="bold"
                )

                draw_numbered_detections(
                    axes[1], image_tensor, hybrid_det, sel_indices, class_names
                )
                axes[1].set_title(
                    "NeSy (FRCNN+SODT) Detections", fontsize=13, fontweight="bold"
                )

                draw_ground_truth_boxes(
                    axes[2], image_tensor, gt_boxes, gt_labels, class_names
                )
                axes[2].set_title("Ground Truth", fontsize=13, fontweight="bold")
            else:
                fig, axes = plt.subplots(1, 2, figsize=(14, 6))

                draw_numbered_detections(
                    axes[0], image_tensor, frcnn_det, sel_indices, class_names
                )
                axes[0].set_title(
                    "Faster R-CNN Detections", fontsize=13, fontweight="bold"
                )

                draw_numbered_detections(
                    axes[1], image_tensor, hybrid_det, sel_indices, class_names
                )
                axes[1].set_title(
                    "NeSy (FRCNN+SODT) Detections", fontsize=13, fontweight="bold"
                )

            plt.tight_layout()
            plt.show()

            # ═══════════════════════════════════════════
            # EXPLANATION SECTION: per-detection buttons
            # ═══════════════════════════════════════════
            if not sel_indices:
                print("No detections above threshold — skipping explanations.")
                return

            print("── Explanation ──")

            def make_int_btn(btn_num, det_idx, expl, gcam_res, tnsr, hyb):
                label_name = class_names[expl["label"] - 1]
                score = expl["score"]
                btn = widgets.Button(
                    description=f"#{btn_num} {label_name} {score:.2f}",
                    layout=widgets.Layout(width="180px"),
                )

                def on_click(
                    _,
                    _tnsr=tnsr,
                    _hyb=hyb,
                    _det_idx=det_idx,
                    _expl=expl,
                    _gcam_res=gcam_res,
                    _btn_num=btn_num,
                ):
                    with expl_output:
                        clear_output(wait=True)

                        def gcam_panel(ax):
                            _int_draw_gradcam_panel(ax, _tnsr, _gcam_res, class_names)

                        draw_neurosymbolic_explanation(
                            _tnsr,
                            _hyb,
                            _det_idx,
                            _expl,
                            class_names,
                            hybrid_model.symbolic_tree,
                            selected_number=_btn_num,
                            extra_panel_func=gcam_panel,
                        )

                btn.on_click(on_click)
                return btn

            buttons = []
            for btn_num, (det_idx, expl, gcam_res) in enumerate(
                zip(sel_indices, sel_explanations, gcam_results), 1
            ):
                buttons.append(
                    make_int_btn(
                        btn_num, det_idx, expl, gcam_res, image_tensor, hybrid_det
                    )
                )

            display(widgets.HBox(buttons))
            display(expl_output)
    finally:
        _processing_upload = False


int_upload.observe(_on_upload_change, names="value")

display(
    widgets.VBox(
        [
            widgets.HTML("<b>Upload a PCB image — inference runs automatically</b>"),
            int_upload,
            int_output,
        ]
    )
)

## 7. Cleanup

In [ ]:
gradcam.release()
print("Grad-CAM hooks released.")